#### QUERY AND VALIDATE `GIZMO.BRONZE.ORDERS`

In [0]:
orders_df = spark.table('''GIZMO.BRONZE.ORDERS_VW''')
display(orders_df)

#### PARSE ARRAY TYPE FOR `ORDERS`
- EXTRACT AND CAST KEY ORDER FIELDS FROM RAW JSON DATA

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, TimestampType

# 2. Apply the regex fix to the 'value' column first
# This addresses the order_date formatting issue
fixed_order_date_df = orders_df.withColumn(
    "fixed_value", 
    F.regexp_replace(F.col("value"), '"order_date": (\d{4}-\d{2}-\d{2})', '"order_date": "$1"')
)

# 3. Extract fields from the JSON string
# Note: Spark's get_json_object returns strings, so we cast to the desired types
fixed_order_df = fixed_order_date_df.select(
    F.get_json_object("value", "$.order_id").cast(IntegerType()).alias("order_id"),
    F.get_json_object("value", "$.customer_id").cast(IntegerType()).alias("customer_id"),
    F.col("fixed_value"),
    F.get_json_object("value", "$.transaction_timestamp").cast(TimestampType()).alias("transaction_timestamp"),
    F.get_json_object("value", "$.total_amount").cast(DoubleType()).alias("total_amount"),
    F.get_json_object("value", "$.payment_method").alias("payment_method"),
    F.get_json_object("value", "$.items[0].item_id").cast(IntegerType()).alias("item_1"),
    F.get_json_object("value", "$.items[1].item_id").cast(IntegerType()).alias("item_2")
)

display(fixed_order_df)

In [0]:
from pyspark.sql.types import DateType

fixed_order_df = fixed_order_date_df.select(
    F.get_json_object("value", "$.order_id").cast(IntegerType()).alias("order_id"),
    F.get_json_object("value", "$.customer_id").cast(IntegerType()).alias("customer_id"),
    
    # Extracting from the repaired JSON column
    F.get_json_object("fixed_value", "$.order_date").cast(DateType()).alias("order_date"),
    
    F.col("fixed_value"),
    F.get_json_object("value", "$.transaction_timestamp").cast(TimestampType()).alias("transaction_timestamp"),
    F.get_json_object("value", "$.total_amount").cast(DoubleType()).alias("total_amount"),
    F.get_json_object("value", "$.payment_method").alias("payment_method"),
    F.get_json_object("value", "$.items[0].item_id").cast(IntegerType()).alias("first_item_id")
)

display(fixed_order_df)

In [0]:
from pyspark.sql import functions as F

# 1. Define the schema
items_schema = "array<struct<item_id:int,quantity:int,price:double>>"

# 2. Transform
# Note: Included 'value' in the select so withColumn can use it
fixed_order_items_df = (
    fixed_order_df.select(
        'order_id', 
        'customer_id', 
        'order_date', 
        'fixed_value', 
        'transaction_timestamp',
        'total_amount', 
        'payment_method'
    )
    .withColumn(
        "items", 
        F.from_json(F.get_json_object(F.col("fixed_value"), "$.items"), items_schema)
    )
)

display(fixed_order_items_df)

#### WRITE TRANSFORMED DATA TO SILVER SCHEMA
1. CATALOG NAME: GIZMO
2. SCHEMA NAME: SILVER
3. TABLE NAME: ORDERS

In [0]:
# Create a temporary view named "v_fixed_orders"
fixed_order_items_df.createOrReplaceTempView("fixed_orders_temp_vw")

# You can now query it using Spark SQL
fixed_orders_temp_vw_df = spark.sql("SELECT * FROM fixed_orders_temp_vw")
display(fixed_orders_temp_vw_df)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW ORDERS_VW_TEMP_VW
AS
SELECT
  from_json(
    fixed_value, 
    'struct<
      items:array<struct<item_id:bigint,name:string,price:bigint,quantity:bigint>>,
      customer_id:int,
      order_date:string,
      order_id:bigint,
      order_status:string,
      payment_method:string,
      total_amount:bigint,
      transaction_timestamp:string
    >'
  ) AS json_value
FROM
  fixed_orders_temp_vw;

#### WRITE TRANSFORMED DATA TO SILVER SCHEMA
1. CATALOG NAME: GIZMO
2. SCHEMA NAME: SILVER
3. TABLE NAME: ORDERS

In [0]:
%sql
CREATE  OR REPLACE TABLE GIZMO.SILVER.ORDERS_JSON_DELTA
AS
SELECT * FROM ORDERS_VW_TEMP_VW;

In [0]:
%sql
SELECT
json_value.order_id::int AS order_id,
json_value.customer_id::int AS customer_id,
json_value.order_date::date AS order_date,
json_value.transaction_timestamp::timestamp AS transaction_timestamp,
json_value.total_amount::int AS total_amount,
json_value.payment_method::string AS payment_status,
explode(array_distinct(json_value.items)) AS item
FROM
GIZMO.SILVER.ORDERS_JSON_DELTA
ORDER BY 1;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW ORDERS_ITEM_EXPLODE_TEMP_VW
AS
SELECT
json_value.customer_id::INT AS customer_id,
json_value.order_id::INT AS order_id,
json_value.order_date::DATE AS order_date,
json_value.order_status::STRING AS order_status,
json_value.payment_method::STRING AS payment_method,
json_value.total_amount::INT AS total_amount,
json_value.transaction_timestamp::TIMESTAMP AS transaction_timestamp,
explode(array_distinct(json_value.items)) AS item
FROM
GIZMO.SILVER.ORDERS_JSON_DELTA;

#### WRITE TRANSFORMED DATA TO SILVER SCHEMA
1. CATALOG NAME: GIZMO
2. SCHEMA NAME: SILVER
3. TABLE NAME: ORDERS_DELTA


In [0]:
%sql
CREATE  OR REPLACE TABLE GIZMO.SILVER.ORDERS_DELTA
AS
SELECT
order_id,
customer_id,
item.item_id,
item.name,
order_date,
order_status,
item.price,
item.quantity,
payment_method,
total_amount,
transaction_timestamp
 FROM ORDERS_ITEM_EXPLODE_TEMP_VW;

#### VALIDATE AND QUERY `GIZMO.SILVER.ORDERS`

In [0]:
%sql
SELECT * FROM GIZMO.SILVER.ORDERS_DELTA LIMIT 15;

In [0]:
%python
dbutils.notebook.exit("ORDERS LOADED INTO GIZMO.SILVER.ORDERS_DELTA")